# 带有 Milvus 矢量数据库的图 RAG

## 概述

### 你将学到什么
本笔记本演示了一种创新的**图 RAG（检索增强生成）** 方法，它将知识图谱与向量数据库的强大功能相结合，可显着提高问答性能，尤其是对于复杂的多跳查询。在本教程结束时，您将了解如何构建一个 Graph RAG 系统，该系统可以回答需要多个逻辑步骤和关系遍历的问题。

### 问题：传统 RAG 的局限性
传统的 RAG 系统虽然功能强大，但仍面临以下问题：
- **多跳问题**：需要多个逻辑步骤的查询（例如，“欧拉老师的儿子做了什么贡献？”）
- **复杂的实体关系**：了解不同实体如何相互连接和关联
- **上下文碎片**：重要的关系可能分散在不同的文本段落中
- **语义差距**：简单的相似性搜索可能会错过逻辑相关但语义遥远的信息

### 解决方案：带有向量数据库的图 RAG
该笔记本提出了一种**统一的方法**，仅使用**矢量数据库**（Milvus）即可实现 Graph RAG 功能，无需单独的图形数据库，同时保持卓越的性能。以下是这种方法的特别之处：

**关键创新**：我们没有存储显式的图结构，而是将实体和关系嵌入为向量，并使用智能检索和扩展技术来重建类似图的推理路径。

### 主要优点
1. **简化架构**：单一矢量数据库而不是矢量数据库+图数据库组合
2. **卓越的多跳性能**：处理需要多个关系遍历的复杂查询
3. **可扩展**：利用 Milvus 的分布式架构进行十亿级部署
4. **经济高效**：降低基础设施复杂性和运营开销
5. **灵活**：适用于任何文本语料库 - 只需提取实体和关系

### 方法概述
我们的方法包括四个主要阶段：

1. **离线数据准备**
   - 从文本语料库中提取实体和关系（三元组）
   - 创建三个向量集合：实体、关系和段落
   - 建立实体和关系之间的邻接映射

2. **查询时检索**
   - 使用向量相似性搜索检索相似的实体和关系
   - 使用命名实体识别（NER）来识别查询实体

3. **子图扩展** 
   - 使用邻接矩阵将检索到的实体/关系扩展到其邻域
   - 支持多度扩展（1跳、2跳邻居）
   - 合并实体和关系扩展路径的结果

4. **法学硕士重新排名**
   - 使用大型语言模型对候选关系进行智能过滤和排名
   - 应用思维链推理来选择最相关的关系
   - 返回最终段落以生成答案

### 架构图
下图展示了完整的工作流程：

![](../images/graph_rag_with_milvus_1.png)

**为什么这样做**：通过将实体和关系表示为向量，我们可以利用语义相似性进行初始检索，然后使用图论扩展来发现间接关系，最后应用 LLM 推理来过滤相关性。这创建了一个“两全其美”的系统，结合了语义搜索、图遍历和智能推理。

---

## 技术实现

在本节中，我们将实现方法概述中描述的 Graph RAG 系统。该实现遵循我们的四阶段方法：数据准备、向量存储、查询处理和智能重新排名。

## 先决条件

要完成此演示，您需要一个矢量数据库。您可以通过[注册 Zilliz Cloud](https://cloud.zilliz.com/signup?utm_source=github&utm_medium=referral&utm_campaign=Nir-250512) 免费获得完全托管的 Milvus 矢量数据库。 Milvus 是一个开源向量数据库，提供高性能的向量相似度搜索。

安装以下依赖项：

In [1]:
! pip install --upgrade --quiet pymilvus numpy scipy langchain langchain-core langchain-openai tqdm

> 如果您使用的是 Google Colab，要启用刚刚安装的依赖项，您可能需要 **重新启动运行时**（单击屏幕顶部的“运行时”菜单，然后从下拉菜单中选择“重新启动会话”）。

我们将使用 OpenAI 的模型。您需要准备 [`OPENAI_API_KEY`](https://platform.openai.com/docs/quickstart) 作为环境变量。

In [2]:
import os

os.environ["OPENAI_API_KEY"] = "sk-***********"

导入必要的库和依赖项。

In [3]:
import numpy as np

from collections import defaultdict
from scipy.sparse import csr_matrix
from pymilvus import MilvusClient
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from tqdm import tqdm

在 Zilliz Cloud 页面上查找您的公共端点和令牌（即 API 密钥）。

![](../images/zilliz_interface.png)

In [4]:
# The `uri` and `token` correspond to the Public Endpoint and Token of your Zilliz Cloud (fully-managed Milvus) cluster.
milvus_client = MilvusClient(
    uri="YOUR_ZILLIZ_PUBLIC_ENDPOINT", 
    token="YOUR_ZILLIZ_TOKEN"
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 离线数据加载

### 理解数据模型

在深入实施之前，了解我们如何构建数据以使用向量进行类似图形的推理至关重要。我们的方法将传统的文本文档转换为三个相互关联的组件：

1. **实体**：我们概念图的“节点”——人、地点、概念等。
2. **关系**：连接实体的“边” - 这些是完整的三元组（主语-谓语-宾语）
3. **段落**：提供上下文和详细信息的原始文本文档

**为什么这种结构有效**：通过将实体和关系分离到不同的向量集合中，我们可以针对查询的不同方面执行有针对性的搜索。当用户问“欧拉老师的儿子做了什么贡献？”时，我们可以：
- 查找与“Euler”相关的实体 
- 寻找连接师生和亲子概念的关系
- 展开图表以发现间接联系
- 检索最相关的段落以生成最终答案

### 数据准备

我们将以介绍伯努利族和欧拉关系的纳米数据集为例进行演示。 Nano 数据集包含 4 个段落和一组相应的三元组，其中每个三元组包含一个主语、一个谓语和一个宾语。

**三元组结构**：每个关系都表示为一个三元组[主语、谓语、宾语]。例如：
- `["Jakob Bernoulli", "was the older brother of", "Johann Bernoulli"]` 捕捉家庭关系
- `["Johann Bernoulli", "was a student of", "Leonhard Euler"]` 捕捉教育关系

在实践中，您可以使用任何方法从您自己的自定义语料库中提取三元组。常见的方法包括：
- **命名实体识别 (NER)** + **关系提取** 模型
- **开放信息提取**系统，如 OpenIE
- **大型语言模型**，具有结构化提示
- 针对高精度域的**手动注释**

In [5]:
nano_dataset = [
    {
        "passage": "Jakob Bernoulli (1654–1705): Jakob was one of the earliest members of the Bernoulli family to gain prominence in mathematics. He made significant contributions to calculus, particularly in the development of the theory of probability. He is known for the Bernoulli numbers and the Bernoulli theorem, a precursor to the law of large numbers. He was the older brother of Johann Bernoulli, another influential mathematician, and the two had a complex relationship that involved both collaboration and rivalry.",
        "triplets": [
            ["Jakob Bernoulli", "made significant contributions to", "calculus"],
            [
                "Jakob Bernoulli",
                "made significant contributions to",
                "the theory of probability",
            ],
            ["Jakob Bernoulli", "is known for", "the Bernoulli numbers"],
            ["Jakob Bernoulli", "is known for", "the Bernoulli theorem"],
            ["The Bernoulli theorem", "is a precursor to", "the law of large numbers"],
            ["Jakob Bernoulli", "was the older brother of", "Johann Bernoulli"],
        ],
    },
    {
        "passage": "Johann Bernoulli (1667–1748): Johann, Jakob’s younger brother, was also a major figure in the development of calculus. He worked on infinitesimal calculus and was instrumental in spreading the ideas of Leibniz across Europe. Johann also contributed to the calculus of variations and was known for his work on the brachistochrone problem, which is the curve of fastest descent between two points.",
        "triplets": [
            [
                "Johann Bernoulli",
                "was a major figure of",
                "the development of calculus",
            ],
            ["Johann Bernoulli", "was", "Jakob's younger brother"],
            ["Johann Bernoulli", "worked on", "infinitesimal calculus"],
            ["Johann Bernoulli", "was instrumental in spreading", "Leibniz's ideas"],
            ["Johann Bernoulli", "contributed to", "the calculus of variations"],
            ["Johann Bernoulli", "was known for", "the brachistochrone problem"],
        ],
    },
    {
        "passage": "Daniel Bernoulli (1700–1782): The son of Johann Bernoulli, Daniel made major contributions to fluid dynamics, probability, and statistics. He is most famous for Bernoulli’s principle, which describes the behavior of fluid flow and is fundamental to the understanding of aerodynamics.",
        "triplets": [
            ["Daniel Bernoulli", "was the son of", "Johann Bernoulli"],
            ["Daniel Bernoulli", "made major contributions to", "fluid dynamics"],
            ["Daniel Bernoulli", "made major contributions to", "probability"],
            ["Daniel Bernoulli", "made major contributions to", "statistics"],
            ["Daniel Bernoulli", "is most famous for", "Bernoulli’s principle"],
            [
                "Bernoulli’s principle",
                "is fundamental to",
                "the understanding of aerodynamics",
            ],
        ],
    },
    {
        "passage": "Leonhard Euler (1707–1783) was one of the greatest mathematicians of all time, and his relationship with the Bernoulli family was significant. Euler was born in Basel and was a student of Johann Bernoulli, who recognized his exceptional talent and mentored him in mathematics. Johann Bernoulli’s influence on Euler was profound, and Euler later expanded upon many of the ideas and methods he learned from the Bernoullis.",
        "triplets": [
            [
                "Leonhard Euler",
                "had a significant relationship with",
                "the Bernoulli family",
            ],
            ["leonhard Euler", "was born in", "Basel"],
            ["Leonhard Euler", "was a student of", "Johann Bernoulli"],
            ["Johann Bernoulli's influence", "was profound on", "Euler"],
        ],
    },
]

我们构建实体和关系如下：
- 实体是三元组中的主语或宾语，因此我们直接从三元组中提取它们。
- 在这里，我们通过直接连接主语、谓语和宾语（中间有空格）来构建关系的概念。

我们还准备一个字典来将实体 id 映射到关系 id，并准备另一个字典将关系 id 映射到段落 id 供以后使用。

### 构建知识图谱结构

下一步将我们的三元组转换为可搜索的向量格式，同时保留图的连接信息。这个过程涉及几个关键决策：

**实体提取策略**：我们通过从三元组中收集所有主题和对象来提取独特的实体。这确保我们捕获任何关系中提到的每个实体，从而全面覆盖我们的知识领域。

**关系表示**：我们不是将关系存储为单独的主谓宾组件，而是将它们连接成自然语言句子。例如，`["Jakob Bernoulli", "was the older brother of", "Johann Bernoulli"]` 变为 `"Jakob Bernoulli was the older brother of Johann Bernoulli"`。这种方法有几个优点：
- **语义丰富性**：完整的句子为向量嵌入提供了更多上下文
- **自然语言兼容性**：法学硕士可以轻松理解和推理完整的句子
- **降低复杂性**：无需管理单独的谓词词汇表

**邻接映射构建**：我们构建两个关键的映射结构：
1. **`entityid_2_relationids`**：将每个实体映射到它参与的所有关系（启用实体到关系的扩展）
2. **`relationid_2_passageids`**：将每个关系映射到它出现的段落（启用关系到段落检索）

这些映射对于子图扩展过程至关重要，使我们能够在查询期间有效地遍历概念图。

In [6]:
entityid_2_relationids = defaultdict(list)
relationid_2_passageids = defaultdict(list)

entities = []
relations = []
passages = []
for passage_id, dataset_info in enumerate(nano_dataset):
    passage, triplets = dataset_info["passage"], dataset_info["triplets"]
    passages.append(passage)
    for triplet in triplets:
        if triplet[0] not in entities:
            entities.append(triplet[0])
        if triplet[2] not in entities:
            entities.append(triplet[2])
        relation = " ".join(triplet)
        if relation not in relations:
            relations.append(relation)
            entityid_2_relationids[entities.index(triplet[0])].append(
                len(relations) - 1
            )
            entityid_2_relationids[entities.index(triplet[2])].append(
                len(relations) - 1
            )
        relationid_2_passageids[relations.index(relation)].append(passage_id)

### 数据插入

为实体、关系和通道创建 Milvus 集合。我们创建了三个独立的 Milvus 集合，每个集合都针对不同类型的检索进行了优化：

1. **实体集合**：存储实体名称和描述的向量嵌入
   - **目的**：启用以实体为中心的查询，例如“查找与“Euler”类似的实体”
   - **搜索模式**：与查询实体的直接语义相似性

2. **关系集合**：存储完整关系句子的向量嵌入  
   - **目的**：捕获与查询意图匹配的关系中的语义模式
   - **搜索模式**：查找语义上与整个查询相似的关系

3. **Passage Collection**：存储原始文本段落的向量嵌入
   - **目的**：为最终答案提供比较基线和详细背景
   - **搜索模式**：传统的 RAG 式文档检索

**为什么是三个集合？**这种分离允许**多模式检索**：
- 如果查询提到特定实体，我们将通过实体集合进行检索
- 如果查询描述了关系或操作，我们通过关系集合进行检索  
- 我们可以结合两条路径的结果并与传统的段落检索进行比较

**嵌入一致性**：所有集合都使用相同的嵌入模型，以确保相似性搜索和结果合并期间的兼容性。

In [7]:
embedding_dim = len(embedding_model.embed_query("foo"))


def create_milvus_collection(collection_name: str):
    """
    Create a new Milvus collection with specified configuration.
    
    This function creates a new Milvus collection for storing vector embeddings.
    If a collection with the same name already exists, it will be dropped first
    to ensure a clean state.
    
    Args:
        collection_name (str): The name of the collection to create.
    """
    if milvus_client.has_collection(collection_name=collection_name):
        milvus_client.drop_collection(collection_name=collection_name)
    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        consistency_level="Strong",
    )


entity_col_name = "entity_collection"
relation_col_name = "relation_collection"
passage_col_name = "passage_collection"
create_milvus_collection(entity_col_name)
create_milvus_collection(relation_col_name)
create_milvus_collection(passage_col_name)

将数据及其元数据信息插入到 Milvus 集合中，包括实体、关系和段落集合。元数据信息包括段落id和邻接实体或关系id。

In [8]:
def milvus_insert(
    collection_name: str,
    text_list: list[str],
):
    """
    Insert text data with embeddings into a Milvus collection in batches.
    
    This function processes a list of text strings, generates embeddings for them,
    and inserts the data into the specified Milvus collection in batches for
    efficient processing.
    
    Args:
        collection_name (str): The name of the Milvus collection to insert data into.
        text_list (list[str]): A list of text strings to be embedded and inserted.
    """
    batch_size = 512
    for row_id in tqdm(range(0, len(text_list), batch_size), desc="Inserting"):
        batch_texts = text_list[row_id : row_id + batch_size]
        batch_embeddings = embedding_model.embed_documents(batch_texts)

        batch_ids = [row_id + j for j in range(len(batch_texts))]
        batch_data = [
            {
                "id": id_,
                "text": text,
                "vector": vector,
            }
            for id_, text, vector in zip(batch_ids, batch_texts, batch_embeddings)
        ]
        milvus_client.insert(
            collection_name=collection_name,
            data=batch_data,
        )


milvus_insert(
    collection_name=relation_col_name,
    text_list=relations,
)

milvus_insert(
    collection_name=entity_col_name,
    text_list=entities,
)

milvus_insert(
    collection_name=passage_col_name,
    text_list=passages,
)

Inserting: 100%|███████████████████████████████████| 1/1 [00:00<00:00,  2.28it/s]


## 在线查询

### 了解查询处理管道

查询阶段实现了我们的核心创新：将语义向量搜索与图遍历逻辑相结合。这个多阶段过程通过以下步骤将自然语言问题转化为相关知识：

1. **实体识别**：使用NER提取查询中提到的实体
2. **双重检索**：同时搜索实体和关系集合  
3. **图扩展**：利用邻接信息发现间接连接
4. **LLM Reranking**：应用智能过滤来选择最相关的关系
5. **答案生成**：检索最终段落并生成响应

### 相似度检索

我们根据 Milvus 的输入查询检索前 K 个相似实体和关系。

在执行实体检索时，我们应该首先使用诸如 NER（命名实体识别）之类的特定方法从查询文本中提取查询实体。为了简单起见，我们在这里准备 NER 结果。如果要将查询更改为自定义问题，则必须更改相应的查询 NER 列表。
实际上，您可以使用任何其他模型或方法从查询中提取实体。

### 双路径检索策略

我们的方法执行两个并行的相似性搜索：

**路径 1：基于实体的检索**
- **输入**：从查询中提取实体（使用 NER）  
- **流程**：在我们的知识库中查找与查询实体类似的实体
- **为什么选择 NER？**：许多复杂的查询引用特定的实体（“Euler”、“Bernoulli family”）。通过明确识别这些，我们可以找到直接匹配项及其关联关系
- **示例**：对于“欧拉老师的儿子做出了什么贡献？”，NER 将“欧拉”识别为关键实体

**路径 2：基于关系的检索**  
- **输入**：完整的查询文本
- **流程**：查找语义上与查询意图匹配的关系
- **目的**：捕获关系模式和问题结构
- **示例**：查询模式“X老师的儿子做出的贡献”与有关家庭关系和贡献的关系模式匹配

**双重检索的好处**：
- **全面覆盖**：实体路径捕获直接提及，关系路径捕获语义模式
- **鲁棒性冗余**：如果一条路径丢失了相关信息，另一条路径可能会捕获它
- **不同的粒度**：实体提供特定的锚点，关系提供结构模式

In [9]:
query = "What contribution did the son of Euler's teacher make?"

query_ner_list = ["Euler"]
# query_ner_list = ner(query) # In practice, replace it with your custom NER approach

query_ner_embeddings = [
    embedding_model.embed_query(query_ner) for query_ner in query_ner_list
]

top_k = 3

entity_search_res = milvus_client.search(
    collection_name=entity_col_name,
    data=query_ner_embeddings,
    limit=top_k,
    output_fields=["id"],
)

query_embedding = embedding_model.embed_query(query)

relation_search_res = milvus_client.search(
    collection_name=relation_col_name,
    data=[query_embedding],
    limit=top_k,
    output_fields=["id"],
)[0]

### 展开子图

我们使用检索到的实体和关系来扩展子图并获得候选关系，然后从两种方式合并它们。下面是子图扩展过程的流程图：
![](../images/graph_rag_with_milvus_2.png)

这里我们构造一个邻接矩阵，并使用矩阵乘法来计算几度内的邻接映射信息。这样我们就可以快速获取任意程度的扩展信息。

### 图扩展的数学

子图扩展步骤是我们的方法真正发挥作用的地方。我们不存储显式的图数据库，而是使用**邻接矩阵**和**矩阵乘法**来有效计算多跳关系。这种数学方法有几个优点：

**邻接矩阵构造**：我们创建一个二元矩阵，其中如果实体 `i` 参与关系 `j`，则为 `entity_relation_adj[i][j] = 1`，否则为 0。这种稀疏表示捕获了整个图结构。

**通过矩阵幂进行多度扩展**：
- **1度扩展**：`entity_adj_1_degree = entity_relation_adj @ entity_relation_adj.T`
- **2度扩展**：`entity_adj_2_degree = entity_adj_1_degree @ entity_adj_1_degree`  
- **n 度展开**：通过将 1 度矩阵提高 n 次方来计算

**为什么有效**：矩阵乘法自然地实现了图遍历。当我们乘以邻接矩阵时，我们正在计算穿过该图的路径：
- 1-hop：直接连接的实体/关系
- 2-hop：通过一个中间实体连接的实体  
- n-hop：通过（n-1）个中间步骤连接的实体

**计算效率**：使用稀疏矩阵和矢量化运算，我们可以在几毫秒内扩展包含数千个实体的子图，使这种方法具有高度可扩展性。

**双重扩展策略**：我们从检索到的实体和检索到的关系进行扩展，然后合并结果。这确保了我们捕获相关信息，无论初始检索在实体还是关系方面更成功。

In [10]:
# Construct the adjacency matrix of entities and relations where the value of the adjacency matrix is 1 if an entity is related to a relation, otherwise 0.
entity_relation_adj = np.zeros((len(entities), len(relations)))
for entity_id, entity in enumerate(entities):
    entity_relation_adj[entity_id, entityid_2_relationids[entity_id]] = 1

# Convert the adjacency matrix to a sparse matrix for efficient computation.
entity_relation_adj = csr_matrix(entity_relation_adj)

# Use the entity-relation adjacency matrix to construct 1 degree entity-entity and relation-relation adjacency matrices.
entity_adj_1_degree = entity_relation_adj @ entity_relation_adj.T
relation_adj_1_degree = entity_relation_adj.T @ entity_relation_adj

# Specify the target degree of the subgraph to be expanded.
# 1 or 2 is enough for most cases.
target_degree = 1

# Compute the target degree adjacency matrices using matrix multiplication.
entity_adj_target_degree = entity_adj_1_degree
for _ in range(target_degree - 1):
    entity_adj_target_degree = entity_adj_target_degree @ entity_adj_1_degree.T
relation_adj_target_degree = relation_adj_1_degree
for _ in range(target_degree - 1):
    relation_adj_target_degree = relation_adj_target_degree @ relation_adj_1_degree.T

entity_relation_adj_target_degree = entity_adj_target_degree @ entity_relation_adj

通过从目标度扩展矩阵中获取值，我们可以轻松地从检索到的实体和关系中扩展相应的度，以获得子图的所有关系。

In [4]:
expanded_relations_from_relation = set()
expanded_relations_from_entity = set()

filtered_hit_relation_ids = [
    relation_res["entity"]["id"]
    for relation_res in relation_search_res
]
for hit_relation_id in filtered_hit_relation_ids:
    expanded_relations_from_relation.update(
        relation_adj_target_degree[hit_relation_id].nonzero()[1].tolist()
    )

filtered_hit_entity_ids = [
    one_entity_res["entity"]["id"]
    for one_entity_search_res in entity_search_res
    for one_entity_res in one_entity_search_res
]

for filtered_hit_entity_id in filtered_hit_entity_ids:
    expanded_relations_from_entity.update(
        entity_relation_adj_target_degree[filtered_hit_entity_id].nonzero()[1].tolist()
    )

# Merge the expanded relations from the relation and entity retrieval ways.
relation_candidate_ids = list(
    expanded_relations_from_relation | expanded_relations_from_entity
)

relation_candidate_texts = [
    relations[relation_id] for relation_id in relation_candidate_ids
]

我们通过扩展子图得到了候选关系，下一步将由LLM重新排序。

### 法学硕士重新排名

在这个阶段，我们部署LLM强大的自注意力机制来进一步过滤和细化候选关系集。子图扩展步骤为我们提供了许多潜在的相关关系，但并非所有关系对于回答我们的特定查询都同样有用。这就是**大型语言模型**的优势 - 它们可以理解查询和候选关系的语义含义，然后智能地选择最相关的关系。

**为什么需要LLM重新排名**：
- **语义理解**：法学硕士可以理解纯相似性搜索可能会错过的复杂查询意图
- **多跳推理**：法学硕士可以跟踪多个关系之间的逻辑连接
- **上下文感知**：法学硕士考虑关系如何协同工作来回答查询
- **质量过滤**：法学硕士可以识别并优先考虑信息最丰富的关系

**思路链提示策略**：
我们采用结构化方法鼓励法学硕士：
1. **分析查询**：分解回答问题需要哪些信息
2. **识别关键联系**：确定哪种类型的关系最有帮助  
3. **有关相关性的原因**：解释为什么选择特定关系
4. **按重要性排名**：根据最终答案的效用对关系进行排序

**一次性学习模式**：我们提供推理过程的具体示例来指导法学硕士的行为。此示例演示如何识别核心实体、跟踪多跳连接以及确定最直接关系的优先级。

**JSON 输出格式**：通过要求结构化 JSON 输出，我们确保可靠的解析和一致的结果，使系统在生产使用中保持稳健。

#### 定义一次性学习示例

首先，我们准备一次性学习示例来指导 LLM 的推理过程：

In [11]:
query_prompt_one_shot_input = """I will provide you with a list of relationship descriptions. Your task is to select 3 relationships that may be useful to answer the given question. Please return a JSON object containing your thought process and a list of the selected relationships in order of their relevance.

Question:
When was the mother of the leader of the Third Crusade born?

Relationship descriptions:
[1] Eleanor was born in 1122.
[2] Eleanor married King Louis VII of France.
[3] Eleanor was the Duchess of Aquitaine.
[4] Eleanor participated in the Second Crusade.
[5] Eleanor had eight children.
[6] Eleanor was married to Henry II of England.
[7] Eleanor was the mother of Richard the Lionheart.
[8] Richard the Lionheart was the King of England.
[9] Henry II was the father of Richard the Lionheart.
[10] Henry II was the King of England.
[11] Richard the Lionheart led the Third Crusade.

"""
query_prompt_one_shot_output = """{"thought_process": "To answer the question about the birth of the mother of the leader of the Third Crusade, I first need to identify who led the Third Crusade and then determine who his mother was. After identifying his mother, I can look for the relationship that mentions her birth.", "useful_relationships": ["[11] Richard the Lionheart led the Third Crusade", "[7] Eleanor was the mother of Richard the Lionheart", "[1] Eleanor was born in 1122"]}"""

#### 创建查询提示模板

接下来，我们定义用于格式化新查询的模板：

In [1]:
query_prompt_template = """Question:
{question}

Relationship descriptions:
{relation_des_str}

"""

#### 实现重排序功能

现在我们实现处理候选关系的核心重排序功能：

In [2]:
def rerank_relations(
    query: str, relation_candidate_texts: list[str], relation_candidate_ids: list[str]
) -> list[int]:
    """
    Rerank candidate relations using LLM to select the most relevant ones for answering a query.
    
    This function uses a large language model with Chain-of-Thought prompting to analyze
    candidate relationships and select the most useful ones for answering the given query.
    It employs a one-shot learning approach with a predefined example to guide the LLM's
    reasoning process.
    
    Args:
        query (str): The input question that needs to be answered.
        relation_candidate_texts (list[str]): List of candidate relationship descriptions.
        relation_candidate_ids (list[str]): List of IDs corresponding to the candidate relations.
        
    Returns:
        list[int]: A list of relation IDs ranked by their relevance to the query.
    """
    relation_des_str = "\n".join(
        map(
            lambda item: f"[{item[0]}] {item[1]}",
            zip(relation_candidate_ids, relation_candidate_texts),
        )
    ).strip()
    rerank_prompts = ChatPromptTemplate.from_messages(
        [
            HumanMessage(query_prompt_one_shot_input),
            AIMessage(query_prompt_one_shot_output),
            HumanMessagePromptTemplate.from_template(query_prompt_template),
        ]
    )
    rerank_chain = (
        rerank_prompts
        | llm.bind(response_format={"type": "json_object"})
        | JsonOutputParser()
    )
    rerank_res = rerank_chain.invoke(
        {"question": query, "relation_des_str": relation_des_str}
    )
    rerank_relation_ids = []
    rerank_relation_lines = rerank_res["useful_relationships"]
    id_2_lines = {}
    for line in rerank_relation_lines:
        id_ = int(line[line.find("[") + 1 : line.find("]")])
        id_2_lines[id_] = line.strip()
        rerank_relation_ids.append(id_)
    return rerank_relation_ids

#### 执行重新排名过程

最后，我们将重新排序函数应用于候选关系：

In [3]:
rerank_relation_ids = rerank_relations(
    query,
    relation_candidate_texts=relation_candidate_texts,
    relation_candidate_ids=relation_candidate_ids,
)

### 获取最终结果

我们可以从重新排序的关系中得到最终检索到的段落。最后一步通过直接与传统 RAG 方法进行比较来展示我们的 Graph RAG 方法的强大功能。这一比较揭示了为什么基于图的推理对于复杂的多跳问题至关重要。

**我们的方法 - 图 RAG 过程**：
1. 从LLM过滤的重新排序关系开始
2. 使用 `relationid_2_passageids` 将关系映射回其源段落
3. 收集独特的段落，同时保留相关顺序
4. 返回top-k最相关的段落用于答案生成

**基线 - 朴素 RAG 过程**：
1. 使用查询嵌入直接搜索段落集合
2. 返回top-k语义最相似的段落
3. 没有考虑实体关系或图结构

**主要区别**：
- **Graph RAG**：通过实体关系推理找到相关上下文
- **Naive RAG**：仅依赖于查询和段落之间的表面语义相似性

**预期结果**：对于像“欧拉老师的儿子做出了什么贡献？”这样的多跳问题，我们的 Graph RAG 方法应该：
- **识别推理链**：欧拉→约翰·伯努利（老师）→丹尼尔·伯努利（儿子）→贡献
- **检索相关段落**：查找有关丹尼尔·伯努利对流体动力学的贡献的段落
- **提供准确的答案**：根据正确的上下文信息生成响应

相比之下，幼稚的 RAG 可能会直接检索有关欧拉的段落，或者完全错过多跳连接，从而导致不完整或不正确的答案。

In [12]:
final_top_k = 2

final_passages = []
final_passage_ids = []
for relation_id in rerank_relation_ids:
    for passage_id in relationid_2_passageids[relation_id]:
        if passage_id not in final_passage_ids:
            final_passage_ids.append(passage_id)
            final_passages.append(passages[passage_id])
passages_from_our_method = final_passages[:final_top_k]

我们可以将结果与朴素的 RAG 方法进行比较，该方法根据直接从段落集合中嵌入的查询来检索 topK 段落。

In [13]:
naive_passage_res = milvus_client.search(
    collection_name=passage_col_name,
    data=[query_embedding],
    limit=final_top_k,
    output_fields=["text"],
)[0]
passages_from_naive_rag = [res["entity"]["text"] for res in naive_passage_res]

print(
    f"Passages retrieved from naive RAG: \n{passages_from_naive_rag}\n\n"
    f"Passages retrieved from our method: \n{passages_from_our_method}\n\n"
)


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            """Use the following pieces of retrieved context to answer the question. If there is not enough information in the retrieved context to answer the question, just say that you don't know.
Question: {question}
Context: {context}
Answer:""",
        )
    ]
)

rag_chain = prompt | llm | StrOutputParser()

answer_from_naive_rag = rag_chain.invoke(
    {"question": query, "context": "\n".join(passages_from_naive_rag)}
)
answer_from_our_method = rag_chain.invoke(
    {"question": query, "context": "\n".join(passages_from_our_method)}
)

print(
    f"Answer from naive RAG: {answer_from_naive_rag}\n\nAnswer from our method: {answer_from_our_method}"
)

Passages retrieved from naive RAG: 
['Leonhard Euler (1707–1783) was one of the greatest mathematicians of all time, and his relationship with the Bernoulli family was significant. Euler was born in Basel and was a student of Johann Bernoulli, who recognized his exceptional talent and mentored him in mathematics. Johann Bernoulli’s influence on Euler was profound, and Euler later expanded upon many of the ideas and methods he learned from the Bernoullis.', 'Johann Bernoulli (1667–1748): Johann, Jakob’s younger brother, was also a major figure in the development of calculus. He worked on infinitesimal calculus and was instrumental in spreading the ideas of Leibniz across Europe. Johann also contributed to the calculus of variations and was known for his work on the brachistochrone problem, which is the curve of fastest descent between two points.']

Passages retrieved from our method: 
['Leonhard Euler (1707–1783) was one of the greatest mathematicians of all time, and his relationship 

结果表明，从普通 RAG 中检索到的段落遗漏了一条真实的段落，从而导致了错误的答案。

从我们的方法中检索到的段落是正确的，这有助于获得问题的准确答案。

### 主要见解和学习成果

比较结果清楚地证明了我们的 Graph RAG 方法对于多跳推理任务的优越性。让我们分析一下我们已经完成的工作：

**性能分析**：
- **朴素的 RAG 限制**：传统的相似性搜索失败，因为查询“欧拉老师的儿子做了什么贡献？”与有关丹尼尔·伯努利流体动力学贡献的段落没有很高的语义相似性。表面层的关键词匹配得不太好。
- **Graph RAG Success**：我们的方法成功追踪逻辑链：查询提到“Euler”→实体检索找到“Leonhard Euler”→图扩展发现“Johann Bernoulli was Euler's Teacher”→进一步扩展发现“Daniel Bernoulli was Johann's son”→关系过滤识别 Daniel 的贡献→检索正确的段落。

**展示的方法创新**：
1. **Vector-only Graph RAG**：我们仅使用向量数据库实现了图级推理，消除了架构复杂性
2. **多模式检索**：结合基于实体和基于关系的搜索路径提供冗余并提高覆盖范围  
3. **数学图扩展**：稀疏矩阵运算实现大规模高效多跳遍历
4. **LLM支持的过滤**：思想链推理提供了超越简单相似性的智能关系选择

**实际应用**：
这种方法在需要复杂推理的领域表现出色：
- **知识库**：科学文献、历史记录、技术文档
- **企业搜索**：跨互连的业务实体和流程查找信息
- **问答**：学术研究、法律文献分析、医学知识检索
- **内容推荐**：通过关系网络了解用户意图

**可扩展性考虑因素**：
- **矢量数据库扩展**：Milvus 可以通过分布式架构处理数十亿个矢量
- **矩阵计算效率**：稀疏矩阵运算随数据大小对数缩放
- **LLM 推理优化**：重新排序步骤可以并行化并缓存重复模式

本教程演示了即使使用更简单的基础设施组件，也可以通过深思熟虑的系统设计来实现复杂的推理功能。这种功能与简单性的平衡使得该方法对于现实世界的部署非常实用。

## 使用完全托管的 Milvus 扩展您的 Graph RAG 系统

虽然本教程中的示例适用于小型数据集，但在生产环境中使用大规模数据实现 Graph RAG 需要强大的基础设施。 Milvus 是一个可扩展至数十亿级的分布式矢量数据库，使其成为管理大规模矢量数据的值得信赖的选择。在生产中管理自托管 Milvus 集群可能会变得具有挑战性。如果您的首要任务是为 RAG 应用程序开发业务逻辑，Zilliz Cloud 提供完全托管的 Milvus 服务，可以为您处理所有操作复杂性：

![Zilliz 云截图](../images/zilliz_screenshot.png)

- **生产就绪**：内置的高可用性和安全功能对于任务关键型人工智能应用程序至关重要
- **性能提高 10 倍**：与高性能开源 Milvus 相比，其专有的 Cardinal 向量索引引擎提供的性能提高了 10 倍。
- **自动索引**：自动索引功能节省了索引选择和参数调整的精力。
- **降低总拥有成本 (TCO)**：在我们处理扩展、更新和监控时，您可以专注于您的应用程序，并且只需为您使用的灵活定价层付费，与管理自托管 Milvus 集群相比，这可以降低 TCO

[**立即免费试用 Zilliz Cloud →**](https://cloud.zilliz.com/signup?utm_source=github&utm_medium=referral&utm_campaign=Nir-250512)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--graphrag-with-milvus-vectordb)